# Phase 7: Custom CNN (Q43–Q46)

**Questions covered:** Q43 (architecture and why each component), Q44 (parameter counts), Q45 (training and validation curves), Q46 (at least three architectures compared).

**What you will build**
- A configurable PyTorch CNN (`src/cnn_models.py`) and a reusable training loop (`src/training.py`).
- Four architectures that differ in depth, kernel size, normalisation, pooling and dropout.
- A comparison on the **validation** set. The best architecture is saved for Phases 8 and 9.

**Design decisions (so you can defend them)**
- **Input:** 128×128 RGB from the Phase 4 cache. Pixels are scaled to 0–1 and normalised with the **training-set** channel mean/std.
- **Data on the GPU as uint8.** The training set at 128×128 is about 0.86 GB, so the whole set lives on the GPU. No DataLoader workers are needed, which matters because `/dev/shm` is only 0.1 GB.
- **Output:** one logit, `sigmoid` → P(Dog), loss `BCEWithLogitsLoss` (numerically stable binary cross-entropy).
- **Optimiser:** AdamW, learning rate 1e-3, weight decay 1e-4, batch size 128, `ReduceLROnPlateau` on validation loss, **early stopping** on validation loss, best weights restored.
- **Mixed precision:** bfloat16 autocast plus channels-last memory layout, for speed on the H200 slice.
- The **test split is not loaded** here.

I have not measured training time on your MIG slice. The notebook has a `QUICK` switch for a 3-epoch pipeline check, and prints the time per epoch so you can decide.

In [ ]:
import os
for var in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"]:
    os.environ[var] = "8"
import sys, json, time, random, importlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from pathlib import Path

SEED = 42
PROJECT_ROOT = Path("/workspace/sic_final_project")      # change here if the folder moves
sys.path.insert(0, str(PROJECT_ROOT / "src"))
MODEL_DIR = PROJECT_ROOT / "models/cnn"
for d in [MODEL_DIR, PROJECT_ROOT / "results/figures", PROJECT_ROOT / "results/metrics", PROJECT_ROOT / "results/comparisons"]:
    d.mkdir(parents=True, exist_ok=True)

needed = ["data/processed/images_128.npy", "data/processed/labels.npy", "data/splits/split_seed42.csv", "src/evaluation.py"]
missing = [f for f in needed if not (PROJECT_ROOT / f).exists()]
for f in needed:
    print("OK     " if f not in missing else "MISSING", f)
assert not missing, "Run Phases 4 and 5 first."
assert torch.cuda.is_available(), "No GPU visible: stop and check the pod before training."
print("GPU:", torch.cuda.get_device_name(0), f"| {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 1: write `cnn_models.py` and `training.py`

**What each CNN component does (Q43)**
- **Convolution** (`Conv2d`): slides small learnable filters (3×3 or 5×5) over the image. Early filters respond to edges and colour blobs, later ones to textures and parts (ears, eyes). Weights are shared across positions, so a pattern is detected anywhere.
- **Activation (ReLU):** `max(0, x)` adds non-linearity. Without it, stacked convolutions would collapse into one linear operation.
- **Batch normalisation** (optional): rescales activations per batch, which stabilises and speeds up training and acts as mild regularisation.
- **Max pooling (2×2):** keeps the strongest response in each 2×2 window, halving height and width. It shrinks computation and makes the features a bit tolerant to small shifts.
- **Flatten vs Global Average Pooling:** both turn the final feature maps into a vector. Flatten keeps every position (many parameters, risk of overfitting). GAP averages each map to one number (few parameters, more robust).
- **Fully connected layer + Dropout:** combines the features into a decision. Dropout randomly zeroes units during training so the network cannot rely on any single one, which reduces overfitting.
- **Output + loss:** one logit; sigmoid turns it into P(Dog); binary cross-entropy penalises confident wrong answers heavily.

In [ ]:
%%writefile /workspace/sic_final_project/src/cnn_models.py
"""Configurable custom CNN for 2-class (binary) image classification. Output: ONE logit per image (Dog = positive)."""
import torch
import torch.nn as nn


def conv_block(cin, cout, kernel, n_convs, batchnorm):
    layers = []
    for i in range(n_convs):
        layers.append(nn.Conv2d(cin if i == 0 else cout, cout, kernel, padding=kernel // 2, bias=not batchnorm))
        if batchnorm:
            layers.append(nn.BatchNorm2d(cout))
        layers.append(nn.ReLU(inplace=True))
    layers.append(nn.MaxPool2d(2))
    return nn.Sequential(*layers)


class CustomCNN(nn.Module):
    def __init__(self, channels=(32, 64, 128), kernel=3, convs_per_block=1, batchnorm=False,
                 pooling="flatten", hidden=128, dropout=0.3, input_size=128):
        super().__init__()
        channels = tuple(channels)
        blocks, cin = [], 3
        for cout in channels:
            blocks.append(conv_block(cin, cout, kernel, convs_per_block, batchnorm))
            cin = cout
        self.features = nn.Sequential(*blocks)
        if pooling == "gap":
            self.pool, feat = nn.AdaptiveAvgPool2d(1), channels[-1]
        elif pooling == "flatten":
            self.pool, feat = nn.Identity(), channels[-1] * (input_size // 2 ** len(channels)) ** 2
        else:
            raise ValueError("pooling must be 'flatten' or 'gap'")
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(feat, hidden), nn.ReLU(inplace=True),
                                        nn.Dropout(dropout), nn.Linear(hidden, 1))

    def forward(self, x):
        return self.classifier(self.pool(self.features(x))).squeeze(1)


ARCHITECTURES = {
    "CNN-A": dict(channels=(32, 64, 128), kernel=3, convs_per_block=1, batchnorm=False, pooling="flatten", hidden=128, dropout=0.3),
    "CNN-B": dict(channels=(32, 64, 128, 256), kernel=3, convs_per_block=1, batchnorm=True, pooling="gap", hidden=128, dropout=0.5),
    "CNN-C": dict(channels=(32, 64, 128, 256), kernel=3, convs_per_block=2, batchnorm=True, pooling="gap", hidden=128, dropout=0.4),
    "CNN-D": dict(channels=(32, 64, 128), kernel=5, convs_per_block=1, batchnorm=True, pooling="gap", hidden=128, dropout=0.3),
}
DESCRIPTIONS = {
    "CNN-A": "baseline: 3 blocks (32-64-128), 3x3, no BN, Flatten, dropout 0.3",
    "CNN-B": "deeper: 4 blocks (32-64-128-256), 3x3, BatchNorm, GAP, dropout 0.5",
    "CNN-C": "VGG-style: 4 blocks x 2 convs, 3x3, BatchNorm, GAP, dropout 0.4",
    "CNN-D": "bigger kernels: 3 blocks (32-64-128), 5x5, BatchNorm, GAP, dropout 0.3",
}


def build_custom_cnn(arch, input_size=128):
    """arch: a key of ARCHITECTURES or a dict of CustomCNN keyword arguments."""
    kwargs = ARCHITECTURES[arch] if isinstance(arch, str) else dict(arch)
    return CustomCNN(**kwargs, input_size=input_size)


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def layer_param_table(model):
    rows = []
    for name, mod in model.named_modules():
        own = list(mod.parameters(recurse=False))
        if own:
            rows.append({"layer": name, "type": type(mod).__name__,
                         "weight shape": tuple(own[0].shape), "params": sum(p.numel() for p in own)})
    return rows

In [ ]:
%%writefile /workspace/sic_final_project/src/training.py
"""Training / evaluation utilities for PyTorch image classifiers whose images live on the GPU as uint8 NHWC tensors."""
import copy, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True      # faster; results can differ very slightly between runs


def channel_stats(images_u8, batch=2048):
    """Per-channel mean/std over uint8 NHWC images scaled to 0-1. Call it on TRAINING images only."""
    n, s, ss = 0, np.zeros(3, np.float64), np.zeros(3, np.float64)
    for i in range(0, len(images_u8), batch):
        b = np.asarray(images_u8[i:i + batch], dtype=np.float32) / 255.0
        s += b.sum(axis=(0, 1, 2), dtype=np.float64)
        ss += np.square(b).sum(axis=(0, 1, 2), dtype=np.float64)
        n += b.shape[0] * b.shape[1] * b.shape[2]
    mean = s / n
    return mean.astype(np.float32), np.sqrt(ss / n - mean ** 2).astype(np.float32)


def _stats(mean, std, dev):
    return (torch.as_tensor(mean, dtype=torch.float32, device=dev).view(1, 3, 1, 1),
            torch.as_tensor(std, dtype=torch.float32, device=dev).view(1, 3, 1, 1))


def to_input(xb_u8, mean_t, std_t, augment_fn=None):
    """uint8 NHWC batch -> normalised float NCHW (channels-last memory). Augmentation acts on 0-1 images."""
    x = xb_u8.permute(0, 3, 1, 2).float().div_(255.0)
    if augment_fn is not None:
        x = augment_fn(x)
    return ((x - mean_t) / std_t).contiguous(memory_format=torch.channels_last)


def _snapshot(model):
    """Copy of everything that can change during training: trainable parameters + all buffers (BatchNorm stats)."""
    snap = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
    snap.update({n: b.detach().cpu().clone() for n, b in model.named_buffers()})
    return snap


@torch.no_grad()
def _eval_loss(model, X, y, mean_t, std_t, bs, amp):
    model.eval()
    tot = torch.zeros((), device=X.device); correct = torch.zeros((), device=X.device)
    for i in range(0, len(X), bs):
        xb, yb = to_input(X[i:i + bs], mean_t, std_t), y[i:i + bs]
        with torch.autocast(device_type=X.device.type, dtype=torch.bfloat16, enabled=amp and X.device.type == "cuda"):
            logits = model(xb)
        logits = logits.float()
        tot += F.binary_cross_entropy_with_logits(logits, yb, reduction="sum")
        correct += ((logits > 0) == (yb > 0.5)).sum()
    return float(tot) / len(X), float(correct) / len(X)


def train_model(model, X_train_u8, y_train, X_val_u8, y_val, mean, std, *, epochs=30, batch_size=128, lr=1e-3,
                weight_decay=1e-4, patience=6, scheduler_patience=2, augment_fn=None, param_groups=None,
                amp=True, seed=42, device=None, verbose=True, min_delta=1e-4, ckpt_path=None):
    """X_*_u8: uint8 NHWC tensors on the GPU; y_*: float tensors (0/1) on the GPU.
    Early stopping on validation loss; the best weights are restored at the end.
    NOTE: train_loss/train_acc are running averages DURING the epoch (dropout and augmentation active), so they are
    pessimistic compared with an eval-mode pass over the training data."""
    dev = device or X_train_u8.device
    seed_everything(seed)
    model.to(dev).to(memory_format=torch.channels_last)
    mean_t, std_t = _stats(mean, std, dev)
    groups = param_groups if param_groups is not None else [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(groups, lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=scheduler_patience)
    loss_fn = nn.BCEWithLogitsLoss()
    n = len(X_train_u8)
    gen = torch.Generator(device=dev); gen.manual_seed(seed)
    hist = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc", "lr", "epoch_s"]}
    best_loss, best_epoch, best_snap, bad = float("inf"), 0, None, 0
    t_start = time.perf_counter()
    for ep in range(1, epochs + 1):
        t0 = time.perf_counter()
        model.train()
        perm = torch.randperm(n, device=dev, generator=gen)
        tot = torch.zeros((), device=dev); correct = torch.zeros((), device=dev); seen = 0
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            if len(idx) < 2:                 # BatchNorm cannot train on a single sample
                continue
            xb, yb = to_input(X_train_u8[idx], mean_t, std_t, augment_fn), y_train[idx]
            with torch.autocast(device_type=dev.type, dtype=torch.bfloat16, enabled=amp and dev.type == "cuda"):
                logits = model(xb)
            loss = loss_fn(logits.float(), yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            tot += loss.detach() * len(idx); correct += ((logits.detach() > 0) == (yb > 0.5)).sum(); seen += len(idx)
        tr_loss, tr_acc = float(tot) / seen, float(correct) / seen
        va_loss, va_acc = _eval_loss(model, X_val_u8, y_val, mean_t, std_t, batch_size * 2, amp)
        sched.step(va_loss)
        for k, v in zip(hist.keys(), [tr_loss, tr_acc, va_loss, va_acc, opt.param_groups[0]["lr"], time.perf_counter() - t0]):
            hist[k].append(v)
        improved = va_loss < best_loss - min_delta
        if improved:
            best_loss, best_epoch, best_snap, bad = va_loss, ep, _snapshot(model), 0
        else:
            bad += 1
        if verbose:
            print(f"ep {ep:3d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {va_loss:.4f} acc {va_acc:.4f} | "
                  f"lr {opt.param_groups[0]['lr']:.1e} | {time.perf_counter() - t0:.1f}s{'  *best*' if improved else ''}")
        if bad >= patience:
            if verbose:
                print(f"early stopping: no val-loss improvement for {patience} epochs (best epoch {best_epoch})")
            break
    model.load_state_dict(best_snap, strict=False)          # restore the best weights
    hist.update(best_epoch=best_epoch, best_val_loss=best_loss, epochs_run=len(hist["val_loss"]),
                train_time_s=time.perf_counter() - t_start)
    if ckpt_path is not None:
        torch.save(model.state_dict(), ckpt_path)
    return hist


@torch.no_grad()
def predict_probs(model, images_u8, mean, std, batch_size=256, device=None, amp=True):
    """images_u8: numpy array or torch tensor, uint8 NHWC. Returns a numpy array of P(Dog)."""
    dev = device or next(model.parameters()).device
    model.eval()
    mean_t, std_t = _stats(mean, std, dev)
    out = []
    for s in range(0, len(images_u8), batch_size):
        xb = images_u8[s:s + batch_size]
        xb = xb.to(dev) if torch.is_tensor(xb) else torch.from_numpy(np.ascontiguousarray(xb)).to(dev)
        with torch.autocast(device_type=dev.type, dtype=torch.bfloat16, enabled=amp and dev.type == "cuda"):
            logits = model(to_input(xb, mean_t, std_t))
        out.append(torch.sigmoid(logits.float()).cpu())
    return torch.cat(out).numpy()


def evaluate_dl(model, images_u8, y_true, mean, std, batch_size=256, device=None):
    """Returns (metrics dict, predicted labels, P(Dog), ms per image)."""
    import evaluation
    dev = device or next(model.parameters()).device
    if dev.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    probs = predict_probs(model, images_u8, mean, std, batch_size, dev)
    if dev.type == "cuda":
        torch.cuda.synchronize()
    ms = 1000 * (time.perf_counter() - t0) / len(probs)
    pred = (probs > 0.5).astype(int)
    return evaluation.compute_metrics(np.asarray(y_true), pred, probs), pred, probs, ms


def plot_history(hist, title="", path=None, axes=None):
    own = axes is None
    if own:
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    ep = np.arange(1, len(hist["val_loss"]) + 1)
    axes[0].plot(ep, hist["train_loss"], label="train"); axes[0].plot(ep, hist["val_loss"], label="validation")
    axes[1].plot(ep, hist["train_acc"], label="train"); axes[1].plot(ep, hist["val_acc"], label="validation")
    for a, name in zip(axes, ["loss (BCE)", "accuracy"]):
        a.axvline(hist["best_epoch"], color="gray", ls=":", label=f"best epoch {hist['best_epoch']}")
        a.set(xlabel="epoch", ylabel=name, title=f"{title} - {name}"); a.legend(fontsize=8)
    if own:
        plt.tight_layout()
        if path:
            plt.savefig(path, dpi=110)
        plt.show()

## Step 2: load the data and put it on the GPU

Normalisation statistics come from the **training images only** (no leakage from validation or test).

In [ ]:
import evaluation, cnn_models, training
for m in (evaluation, cnn_models, training):
    importlib.reload(m)
dev = torch.device("cuda")
training.seed_everything(SEED)

split_df = pd.read_csv(PROJECT_ROOT / "data/splits/split_seed42.csv")
labels = np.load(PROJECT_ROOT / "data/processed/labels.npy")
imgs128 = np.load(PROJECT_ROOT / "data/processed/images_128.npy", mmap_mode="r")
tr = split_df.loc[split_df["split"] == "train", "array_index"].to_numpy()
va = split_df.loc[split_df["split"] == "val", "array_index"].to_numpy()
y_train, y_val = labels[tr], labels[va]

mean, std = training.channel_stats(imgs128[tr])
print("training-set channel mean:", mean.round(4), "| std:", std.round(4))

Xtr = torch.from_numpy(np.ascontiguousarray(imgs128[tr])).to(dev)          # uint8 NHWC on the GPU
Xva = torch.from_numpy(np.ascontiguousarray(imgs128[va])).to(dev)
ytr = torch.from_numpy(y_train).float().to(dev); yva = torch.from_numpy(y_val).float().to(dev)
print(f"train {tuple(Xtr.shape)} | val {tuple(Xva.shape)} | GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
sub = np.random.default_rng(SEED).choice(len(tr), 3000, replace=False)       # for eval-mode train accuracy
sub_t = torch.from_numpy(sub).to(dev)

## Q43 and Q44: the baseline architecture and its parameters

The code below prints the baseline (CNN-A), follows one image through the network (shape after each block), and computes the parameter count **by hand** from the formulas, then checks it against PyTorch.

- Conv layer parameters = (kernel × kernel × input channels + 1 bias) × output channels
- Linear layer parameters = inputs × outputs + outputs
- Pooling, ReLU, Flatten and Dropout have **no** parameters.

In [ ]:
model = cnn_models.build_custom_cnn("CNN-A", input_size=128)
print(model)

x = torch.zeros(1, 3, 128, 128)
print("\nShape of one image as it flows through the network:")
print("input".ljust(14), tuple(x.shape))
for i, blk in enumerate(model.features):
    x = blk(x); print(f"block {i + 1}".ljust(14), tuple(x.shape))
x = model.pool(x); print("pool".ljust(14), tuple(x.shape), "-> flattened", x.numel(), "values")

In [ ]:
rows = cnn_models.layer_param_table(model)
table = pd.DataFrame(rows)
# manual formulas for CNN-A
manual = {"conv1": (3 * 3 * 3 + 1) * 32, "conv2": (3 * 3 * 32 + 1) * 64, "conv3": (3 * 3 * 64 + 1) * 128,
          "fc1": 128 * 16 * 16 * 128 + 128, "output": 128 * 1 + 1}
print("Manual calculation:")
for k, v in manual.items():
    print(f"  {k:7s} {v:>10,d}")
print(f"  {'TOTAL':7s} {sum(manual.values()):>10,d}")
display(table)
total, trainable = cnn_models.count_params(model)
print(f"PyTorch count: total {total:,} | trainable {trainable:,}")
assert total == sum(manual.values()), "manual formula and PyTorch disagree: re-check the architecture"
print("Manual count matches PyTorch.")
print("\nQ44 - which layer holds most parameters?", table.sort_values("params", ascending=False).iloc[0]["layer"])

## Q46: four architectures (and training setup for Q45)

| Name | What changes |
|---|---|
| CNN-A | baseline: 3 blocks, no BatchNorm, **Flatten** (many parameters in the first dense layer), dropout 0.3 |
| CNN-B | **deeper** (4 blocks), **BatchNorm**, **global average pooling**, dropout 0.5 |
| CNN-C | **two convolutions per block** (VGG-style), BatchNorm, GAP, dropout 0.4 |
| CNN-D | **5×5 kernels**, 3 blocks, BatchNorm, GAP, dropout 0.3 |

Same data, optimiser, schedule and seed for every model, so differences come from the architecture. Each is trained **once** (one seed), so differences of a few tenths of a percentage point are within noise. Say so when you discuss the results.

Set `QUICK = True` first (3 epochs) to check the pipeline and measure the time per epoch. Then set it to `False` for the real run.

In [ ]:
QUICK = True            # True: 3 epochs to test the pipeline. Set False for the real run.
BATCH, LR, WD = 128, 1e-3, 1e-4
EPOCHS, PATIENCE = (3, 2) if QUICK else (30, 6)
ARCH_NAMES = ["CNN-A", "CNN-B", "CNN-C", "CNN-D"]
print("QUICK" if QUICK else "FULL", "run:", dict(epochs=EPOCHS, patience=PATIENCE, batch=BATCH, lr=LR, weight_decay=WD))
for n in ARCH_NAMES:
    m = cnn_models.build_custom_cnn(n); print(f"{n}: {cnn_models.count_params(m)[0]:>10,d} params | {cnn_models.DESCRIPTIONS[n]}")

In [ ]:
results = {}                    # name -> dict(hist, metrics, probs, ...)

def run_architecture(name):
    model = cnn_models.build_custom_cnn(name)
    ckpt = MODEL_DIR / f"{name.lower().replace('-', '_')}{'_quick' if QUICK else ''}.pt"
    print(f"\n=== {name}: {cnn_models.DESCRIPTIONS[name]} ===")
    hist = training.train_model(model, Xtr, ytr, Xva, yva, mean, std, epochs=EPOCHS, batch_size=BATCH, lr=LR,
                                weight_decay=WD, patience=PATIENCE, seed=SEED, ckpt_path=ckpt)
    metrics, pred, probs, ms = training.evaluate_dl(model, Xva, y_val, mean, std)
    p_tr = training.predict_probs(model, Xtr[sub_t], mean, std)
    train_acc = float(((p_tr > 0.5).astype(int) == y_train[sub]).mean())          # eval mode, 3000 training images
    total, trainable = cnn_models.count_params(model)
    if not QUICK:                                    # quick runs are not logged to the results table
        evaluation.log_result(f"Custom CNN - {name}", "val", metrics, train_accuracy=train_acc,
                              train_time_s=round(hist["train_time_s"], 1), infer_ms_per_img=round(ms, 4),
                              params=total, trainable_params=trainable, epochs_run=hist["epochs_run"],
                              best_epoch=hist["best_epoch"], notes=cnn_models.DESCRIPTIONS[name] + " | GPU")
    results[name] = dict(hist=hist, metrics=metrics, pred=pred, probs=probs, ms=ms, train_acc=train_acc,
                         params=total, ckpt=str(ckpt))
    print(f"-> val acc {metrics['accuracy']:.4f} | AUC {metrics['roc_auc']:.4f} | train acc (eval mode) {train_acc:.4f} | "
          f"best epoch {hist['best_epoch']}/{hist['epochs_run']} | {hist['train_time_s']:.0f}s total | {ms:.3f} ms/img")
    del model; torch.cuda.empty_cache()

run_architecture("CNN-A")

In [ ]:
for name in ["CNN-B", "CNN-C", "CNN-D"]:
    run_architecture(name)

## Q45: training and validation curves

Read the curves like this:
- **Both curves falling, close together:** healthy learning.
- **Training keeps improving while validation stalls or rises:** overfitting (memorising training images).
- **Both stuck at a poor level:** underfitting (model too weak or learning rate too small/large).
The dotted line marks the epoch whose weights were restored (best validation loss).

In [ ]:
fig, axes = plt.subplots(len(results), 2, figsize=(11, 3.4 * len(results)))
axes = np.atleast_2d(axes)
for row, (name, r) in zip(axes, results.items()):
    training.plot_history(r["hist"], title=name, axes=row)
plt.tight_layout(); plt.savefig(PROJECT_ROOT / "results/figures/07_cnn_curves.png", dpi=110); plt.show()

In [ ]:
rows = []
for name, r in results.items():
    m, h = r["metrics"], r["hist"]
    rows.append({"model": name, "params": r["params"], "epochs run": h["epochs_run"], "best epoch": h["best_epoch"],
                 "val acc": m["accuracy"], "val F1": m["f1"], "val AUC": m["roc_auc"], "val loss (best)": h["best_val_loss"],
                 "train acc (eval mode)": r["train_acc"], "train-val gap": r["train_acc"] - m["accuracy"],
                 "train time s": h["train_time_s"], "ms/img": r["ms"]})
cmp = pd.DataFrame(rows).set_index("model")
display(cmp.round(4))

## Choose the best custom architecture (validation set)

Rule: highest **validation accuracy**; if two are within 0.2 percentage points, take the lower **validation loss**. Test data plays no part in this choice.
Skip this cell while `QUICK = True`: a 3-epoch run should not decide anything.

In [ ]:
if QUICK:
    print("QUICK run: nothing is saved as 'best'. Set QUICK = False, rerun the training cells, then run this cell.")
else:
    ranked = cmp.sort_values("val acc", ascending=False)
    top = ranked.index[0]
    near = ranked[ranked["val acc"] >= ranked.loc[top, "val acc"] - 0.002]
    best_name = near["val loss (best)"].idxmin()
    print("Best custom architecture:", best_name, "-", cnn_models.DESCRIPTIONS[best_name])

    best = {"name": best_name, "arch_kwargs": {k: (list(v) if isinstance(v, tuple) else v) for k, v in cnn_models.ARCHITECTURES[best_name].items()},
            "input_size": 128, "mean": [float(v) for v in mean], "std": [float(v) for v in std],
            "ckpt": f"models/cnn/{best_name.lower().replace('-', '_')}.pt",
            "schedule": {"epochs": EPOCHS, "patience": PATIENCE, "batch": BATCH, "lr": LR, "weight_decay": WD, "seed": SEED},
            "val": {k: float(v) for k, v in results[best_name]["metrics"].items()},
            "best_epoch": int(results[best_name]["hist"]["best_epoch"])}
    (MODEL_DIR / "best_custom_cnn.json").write_text(json.dumps(best, indent=2))
    cmp.to_csv(PROJECT_ROOT / "results/comparisons/07_cnn_architectures_val.csv")
    print("saved models/cnn/best_custom_cnn.json")

    from sklearn.metrics import classification_report
    r = results[best_name]
    evaluation.plot_confusion(y_val, r["pred"], f"{best_name} (validation)"); plt.show()
    print(classification_report(y_val, r["pred"], target_names=evaluation.CLASS_NAMES, digits=4))
    for p in sorted(MODEL_DIR.iterdir()):
        print(f"{p.stat().st_size / 1e6:8.1f} MB  {p.name}")

## Your write-ups (use your own numbers and words)

**Q43.** Describe CNN-A layer by layer (use the printed model and the shape table) and the purpose of convolution, ReLU, pooling, flatten/GAP, dense, dropout, the sigmoid output, binary cross-entropy, AdamW, learning rate 1e-3, batch size 128, and the epochs/early-stopping choice.

**Q44.** Give the per-layer parameter table, the total, and which layer dominates (for CNN-A it is the first dense layer behind **Flatten**). Explain why that is a warning sign for overfitting, and why GAP in CNN-B/C/D removes it.

**Q45.** Describe the CNN-A curves: where validation loss bottoms out, the train-vs-validation gap (use the `train-val gap` column), and whether you see overfitting or underfitting. Say what early stopping and the learning-rate schedule did.

**Q46.** Compare the four architectures on validation accuracy, F1, AUC, parameters, time and gap. Which is best and why might that be (depth? BatchNorm? GAP? kernel size?). Be honest that each run is one seed, so small differences may be noise.

**For the report:** inputs 128×128, training-set normalisation, AdamW + ReduceLROnPlateau + early stopping, bfloat16 mixed precision, one seed per architecture, test set untouched.

Files created: `src/cnn_models.py`, `src/training.py`, `models/cnn/*.pt`, `models/cnn/best_custom_cnn.json`, `results/comparisons/07_cnn_architectures_val.csv`, `results/figures/07_cnn_curves.png`, and rows in `results/comparisons/model_results.csv`.